In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [6]:
import sys, os, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score, classification_report,
                             confusion_matrix, precision_recall_curve)
warnings.filterwarnings("ignore")


In [13]:
import os
import glob
import pandas as pd

# Find the uploaded Excel dataset in Kaggle
excel_files = glob.glob("/kaggle/input/**/*.xlsx", recursive=True)

if not excel_files:
    raise FileNotFoundError(
        "No Excel file found. Please attach your dataset using Add Input."
    )

XLSX = excel_files[0]
OUT = "/kaggle/working/churn_output"

os.makedirs(OUT, exist_ok=True)

CUTOFF = pd.Timestamp("2024-03-31")
SEED = 42

print("Dataset:", XLSX)
print("Output folder:", OUT)

Dataset: /kaggle/input/datasets/nagumeena22/churnbakes/FreshBasket_Loyalty_Churn_Dataset.xlsx
Output folder: /kaggle/working/churn_output


In [14]:
prof = pd.read_excel(XLSX, sheet_name="fb Customer Profile", header=1)
act  = pd.read_excel(XLSX, sheet_name="fb Monthly Activity", header=1)
lab  = pd.read_excel(XLSX, sheet_name="fb Churn Label", header=1)
print("raw shapes:", prof.shape, act.shape, lab.shape)

raw shapes: (2608, 10) (28073, 12) (2605, 6)


In [18]:
prof = prof.drop_duplicates("CUSTOMER_ID", keep="first")
lab  = lab.drop_duplicates("CUSTOMER_ID", keep="first")
act  = act.drop_duplicates()                                  
act  = act.drop_duplicates(["CUSTOMER_ID", "MONTH"], keep="first")

In [20]:
for c in ["CITY", "MEMBERSHIP_TIER", "GENDER", "PREFERRED_CATEGORY", "HOME_STORE_PRICE_TIER"]:
    prof[c] = prof[c].astype("string").str.strip().str.title()
prof["GENDER"] = prof["GENDER"].replace({"F": "F", "M": "M", "Other": "Other"})  
prof["GENDER"] = prof["GENDER"].fillna("Unknown")
prof["HOME_STORE_PRICE_TIER"] = prof["HOME_STORE_PRICE_TIER"].fillna("Unknown")

In [21]:
bad = (act.TOTAL_SPEND <= 0) & (act.TRANSACTIONS > 0)
act.loc[bad | (act.TOTAL_SPEND < 0), "TOTAL_SPEND"] = np.nan
act.loc[act.TRANSACTIONS == 0, "TOTAL_SPEND"] = 0.0          

fix = act.TOTAL_SPEND.isna() & (act.AVG_BASKET_VALUE > 0)
act.loc[fix, "TOTAL_SPEND"] = act.AVG_BASKET_VALUE * act.TRANSACTIONS
act["TOTAL_SPEND"] = act.TOTAL_SPEND.fillna(0.0)

cap = act.TOTAL_SPEND.quantile(0.999)
act["TOTAL_SPEND"] = act.TOTAL_SPEND.clip(upper=cap)
act["AVG_BASKET_VALUE"] = np.where(act.TRANSACTIONS > 0, act.TOTAL_SPEND / act.TRANSACTIONS.clip(lower=1), 0)


In [22]:
d = lab.merge(prof, on="CUSTOMER_ID", how="left")
eda = {}
for c in ["MEMBERSHIP_TIER", "GENDER", "CITY", "PREFERRED_CATEGORY", "HOME_STORE_PRICE_TIER", "MARKETING_OPT_IN"]:
    eda[c] = d.groupby(c).CHURNED.agg(["count", "mean"]).rename(columns={"mean": "churn_rate"}).round(3)
monthly = act.groupby("MONTH").agg(active=("TRANSACTIONS", lambda s: (s > 0).sum()),
                                   spend=("TOTAL_SPEND", "sum"))
print("overall churn rate: %.3f" % lab.CHURNED.mean())
for k, v in eda.items(): print("\nChurn by", k); print(v)


overall churn rate: 0.315

Churn by MEMBERSHIP_TIER
                 count  churn_rate
MEMBERSHIP_TIER                   
Gold               824       0.301
Platinum           302       0.311
Silver            1474       0.323

Churn by GENDER
         count  churn_rate
GENDER                    
F         1367       0.327
M         1085       0.298
Other       96       0.281
Unknown     52       0.404

Churn by CITY
           count  churn_rate
CITY                        
Austin       266       0.376
Boise        247       0.320
Columbus     268       0.362
Denver       257       0.288
Madison      256       0.297
Nashville    237       0.333
Portland     240       0.283
Raleigh      282       0.298
Spokane      252       0.290
Tulsa        295       0.298

Churn by PREFERRED_CATEGORY
                    count  churn_rate
PREFERRED_CATEGORY                   
Bakery                338       0.349
Beverages             320       0.306
Dairy                 307       0.303
Frozen Foods

In [40]:
print("Activity columns:")
print(act.columns.tolist())

print("\nActivity sample:")
display(act.head())

print("\nCurrent feature-generation code:")
# Look at the notebook cell above the one that creates F.

Activity columns:
['CUSTOMER_ID', 'MONTH', 'TRANSACTIONS', 'TOTAL_SPEND', 'AVG_BASKET_VALUE', 'DISTINCT_CATEGORIES', 'PROMO_TXN_PCT', 'APP_SESSIONS', 'EMAILS_OPENED', 'COUPONS_REDEEMED', 'SUPPORT_TICKETS', 'COMPLAINT_FLAG']

Activity sample:


,CUSTOMER_ID,MONTH,TRANSACTIONS,TOTAL_SPEND,AVG_BASKET_VALUE,DISTINCT_CATEGORIES,PROMO_TXN_PCT,APP_SESSIONS,EMAILS_OPENED,COUPONS_REDEEMED,SUPPORT_TICKETS,COMPLAINT_FLAG
0,501855,2023-03-01,5,150.01,30.0020,4,0.04,3,0,1,0,0
1,501803,2024-04-01,12,325.77,27.1475,3,0.12,1,2,0,0,0
2,501889,2024-05-01,4,165.54,41.3850,4,0.60,4,1,0,0,0
3,502005,2023-03-01,2,130.82,65.4100,2,0.31,4,1,1,0,0
4,500568,2024-01-01,1,15.86,15.8600,0,0.39,0,0,1,1,1



Current feature-generation code:


In [41]:
hist = act[act.MONTH <= CUTOFF].copy()                         # drop Apr-Jun 2024 completely
def window(df, months, tag):
    start = (CUTOFF.to_period("M") - months + 1).to_timestamp()
    w = df[df.MONTH >= start]
    g = w.groupby("CUSTOMER_ID").agg(
        **{f"txn_{tag}": ("TRANSACTIONS", "sum"), f"spend_{tag}": ("TOTAL_SPEND", "sum"),
           f"app_{tag}": ("APP_SESSIONS", "sum"), f"email_{tag}": ("EMAILS_OPENED", "sum"),
           f"coupon_{tag}": ("COUPONS_REDEEMED", "sum"), f"tickets_{tag}": ("SUPPORT_TICKETS", "sum"),
           f"complaints_{tag}": ("COMPLAINT_FLAG", "sum"), f"cats_{tag}": ("DISTINCT_CATEGORIES", "mean"),
           f"promo_{tag}": ("PROMO_TXN_PCT", "mean"), f"active_m_{tag}": ("TRANSACTIONS", lambda s: (s > 0).sum())})
    return g
F = pd.DataFrame(index=lab.CUSTOMER_ID.unique())
for m, t in [(3, "l3"), (6, "l6"), (12, "l12")]:
    F = F.join(window(hist, m, t))
F = F.join(hist.groupby("CUSTOMER_ID").agg(txn_all=("TRANSACTIONS", "sum"), spend_all=("TOTAL_SPEND", "sum"),
                                           months_hist=("MONTH", "nunique")))

In [42]:
last = hist[hist.TRANSACTIONS > 0].groupby("CUSTOMER_ID").MONTH.max()
F["recency_m"] = ((CUTOFF.year - last.dt.year) * 12 + (CUTOFF.month - last.dt.month))
F["recency_m"] = F.recency_m.fillna(99)                        
F["no_purchase_ever"] = (F.recency_m == 99).astype(int)
# trends / ratios
F["txn_trend_3v6"] = F.txn_l3 / 3 - (F.txn_l6 - F.txn_l3) / 3
F["spend_trend_3v6"] = F.spend_l3 / 3 - (F.spend_l6 - F.spend_l3) / 3
F["txn_ratio_3v12"] = F.txn_l3 / F.txn_l12.replace(0, np.nan)
F["app_trend"] = F.app_l3 / 3 - (F.app_l6 - F.app_l3) / 3
F["email_trend"] = F.email_l3 / 3 - (F.email_l6 - F.email_l3) / 3
F["avg_basket_l6"] = F.spend_l6 / F.txn_l6.replace(0, np.nan)
F["complaint_ever"] = (F.complaints_l12.fillna(0) > 0).astype(int)
F = F.reset_index().rename(columns={"index": "CUSTOMER_ID"})


In [43]:
print("F shape:", F.shape)
print("Duplicate column names in F:")
print(F.columns[F.columns.duplicated()].tolist())

print("CUSTOMER_ID columns in F:",
      (F.columns == "CUSTOMER_ID").sum())

F shape: (2600, 43)
Duplicate column names in F:
[]
CUSTOMER_ID columns in F: 1


In [44]:
F = F.loc[:, ~F.columns.duplicated()].copy()

print("Duplicate columns remaining:",
      F.columns[F.columns.duplicated()].tolist())

,CUSTOMER_ID,txn_l3,spend_l3,app_l3,email_l3,coupon_l3,tickets_l3,complaints_l3,cats_l3,promo_l3,...,months_hist,recency_m,no_purchase_ever,txn_trend_3v6,spend_trend_3v6,txn_ratio_3v12,app_trend,email_trend,avg_basket_l6,complaint_ever
0,500001,21.0,438.28,5.0,1.0,2.0,0.0,0.0,3.666667,0.190000,...,14.0,0.0,0,4.666667,91.753333,0.368421,1.000000,0.000000,21.475000,1
1,500002,12.0,357.11,5.0,1.0,3.0,0.0,0.0,1.000000,0.236667,...,9.0,0.0,0,-2.333333,-91.440000,0.226415,-1.000000,0.000000,31.888387,0
2,500003,14.0,489.49,6.0,1.0,0.0,2.0,1.0,2.000000,0.366667,...,14.0,0.0,0,1.333333,42.853333,0.250000,-1.000000,0.333333,35.434167,1
3,500004,16.0,367.99,10.0,5.0,3.0,0.0,0.0,3.666667,0.140000,...,14.0,0.0,0,2.333333,48.953333,0.271186,-1.000000,1.333333,23.564800,1
4,500005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,99.0,1,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2595,502596,13.0,616.44,8.0,0.0,1.0,0.0,0.0,1.000000,0.296667,...,15.0,0.0,0,-3.000000,-133.383333,0.185714,-0.333333,-0.666667,46.658000,0
2596,502597,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.0,12.0,0,NaN,NaN,NaN,NaN,NaN,NaN,1
2597,502598,8.0,236.11,4.0,2.0,2.0,0.0,0.0,2.500000,0.320000,...,12.0,0.0,0,0.333333,12.586667,0.190476,-3.000000,-1.666667,28.964000,0
2598,502599,26.0,735.18,5.0,5.0,5.0,0.0,0.0,4.000000,0.213333,...,3.0,0.0,0,8.666667,245.060000,1.000000,1.666667,1.666667,28.276154,0


In [45]:
print("Feature columns:", F.columns.tolist())
print(F.head())

print("\nMissing feature values:")
print(F.isna().sum())

Feature columns: ['CUSTOMER_ID', 'txn_l3', 'spend_l3', 'app_l3', 'email_l3', 'coupon_l3', 'tickets_l3', 'complaints_l3', 'cats_l3', 'promo_l3', 'active_m_l3', 'txn_l6', 'spend_l6', 'app_l6', 'email_l6', 'coupon_l6', 'tickets_l6', 'complaints_l6', 'cats_l6', 'promo_l6', 'active_m_l6', 'txn_l12', 'spend_l12', 'app_l12', 'email_l12', 'coupon_l12', 'tickets_l12', 'complaints_l12', 'cats_l12', 'promo_l12', 'active_m_l12', 'txn_all', 'spend_all', 'months_hist', 'recency_m', 'no_purchase_ever', 'txn_trend_3v6', 'spend_trend_3v6', 'txn_ratio_3v12', 'app_trend', 'email_trend', 'avg_basket_l6', 'complaint_ever']
   CUSTOMER_ID  txn_l3  spend_l3  app_l3  email_l3  coupon_l3  tickets_l3  \
0       500001    21.0    438.28     5.0       1.0        2.0         0.0   
1       500002    12.0    357.11     5.0       1.0        3.0         0.0   
2       500003    14.0    489.49     6.0       1.0        0.0         2.0   
3       500004    16.0    367.99    10.0       5.0        3.0         0.0   
4    

In [46]:
print("Profile shape:", prof.shape)
print("Activity shape:", act.shape)
print("Labels shape:", lab.shape)
print("Features shape:", F.shape)
print("Merged dataframe shape:", df.shape)

print("\nCustomer ID samples:")
print("Profile IDs:", prof["CUSTOMER_ID"].head(10).tolist())
print("Activity IDs:", act["CUSTOMER_ID"].head(10).tolist())
print("Label IDs:", lab["CUSTOMER_ID"].head(10).tolist())
print("Feature IDs:", F["CUSTOMER_ID"].head(10).tolist())

print("\nUnique customer IDs:")
print("Profile:", prof["CUSTOMER_ID"].nunique())
print("Labels:", lab["CUSTOMER_ID"].nunique())
print("Features:", F["CUSTOMER_ID"].nunique())

print("\nTarget distribution:")
print(lab["CHURNED"].value_counts(dropna=False))

print("\nMerged target distribution:")
print(df["CHURNED"].value_counts(dropna=False))

Profile shape: (2600, 10)
Activity shape: (28013, 12)
Labels shape: (2600, 6)
Features shape: (2600, 43)
Merged dataframe shape: (0, 55)

Customer ID samples:
Profile IDs: [500001, 500002, 500003, 500004, 500005, 500006, 500007, 500008, 500009, 500010]
Activity IDs: [501855, 501803, 501889, 502005, 500568, 502203, 500453, 501576, 501661, 501785]
Label IDs: [500001, 500002, 500003, 500004, 500005, 500006, 500007, 500008, 500009, 500010]
Feature IDs: [500001, 500002, 500003, 500004, 500005, 500006, 500007, 500008, 500009, 500010]

Unique customer IDs:
Profile: 2600
Labels: 2600
Features: 2600

Target distribution:
CHURNED
0    1782
1     818
Name: count, dtype: int64

Merged target distribution:
Series([], Name: count, dtype: int64)


In [39]:
print("Profile IDs:", prof["CUSTOMER_ID"].head().tolist())
print("Feature IDs:", F["CUSTOMER_ID"].head().tolist())

print(
    "Matching feature IDs:",
    F["CUSTOMER_ID"].isin(prof["CUSTOMER_ID"]).sum()
)

Profile IDs: [500001, 500002, 500003, 500004, 500005]
Feature IDs: [0, 1, 2, 3, 4]
Matching feature IDs: 0


In [47]:
P = prof.copy()

# Calculate customer tenure as of the prediction cutoff
P["tenure_m"] = (
    (CUTOFF - P["SIGNUP_DATE"]).dt.days / 30.44
).clip(lower=0)

P["age_missing"] = P["AGE"].isna().astype(int)

# Remove accidental duplicate column names
P = P.loc[:, ~P.columns.duplicated()].copy()
F = F.loc[:, ~F.columns.duplicated()].copy()

# Confirm each table has one row per customer before merging
print("Duplicate IDs in labels:", lab["CUSTOMER_ID"].duplicated().sum())
print("Duplicate IDs in profile:", P["CUSTOMER_ID"].duplicated().sum())
print("Duplicate IDs in features:", F["CUSTOMER_ID"].duplicated().sum())

# Merge the tables
df = (
    lab[["CUSTOMER_ID", "CHURNED"]]
    .drop_duplicates(subset="CUSTOMER_ID")
    .merge(
        P.drop(columns=["SIGNUP_DATE"]),
        on="CUSTOMER_ID",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        F,
        on="CUSTOMER_ID",
        how="left",
        validate="one_to_one"
    )
)

# Exclude customers with no activity history before the cutoff
df["has_history"] = (df["months_hist"].fillna(0) > 0).astype(int)

excluded = df[df["has_history"] == 0].copy()
df = df[df["has_history"] == 1].copy()

print("Final modelling rows:", len(df))
print("Excluded customers without history:", len(excluded))
print("Duplicate columns:", df.columns[df.columns.duplicated()].tolist())

Duplicate IDs in labels: 0
Duplicate IDs in profile: 0
Duplicate IDs in features: 0
Final modelling rows: 2481
Excluded customers without history: 119
Duplicate columns: []


In [48]:
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd

# 1. Separate target and features
y = df["CHURNED"].astype(int)

drop_cols = [
    "CHURNED", "CUSTOMER_ID", "has_history",
    "PREFERRED_STORE_ID", "CITY"
]
X = df.drop(columns=drop_cols, errors="ignore").copy()

# 2. Identify categorical and numerical columns
cat = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
num = [c for c in X.columns if c not in cat]

# Convert categorical values to strings, preserving missing values
for c in cat:
    X[c] = X[c].fillna("Missing").astype(str)

# 3. Preprocessing
pre_lr = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
])

pre_rf = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
])

# 4. Models
models = {
    "LogisticRegression": Pipeline([
        ("preprocess", pre_lr),
        ("model", LogisticRegression(
            max_iter=2000,
            C=2.0,
            class_weight="balanced",
            random_state=SEED
        ))
    ]),
    "RandomForest": Pipeline([
        ("preprocess", pre_rf),
        ("model", RandomForestClassifier(
            n_estimators=150,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        ))
    ])
}

# 5. Split the data
Xtr, Xte, ytr, yte = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# 6. Compare models
results = {}

for name, model in models.items():
    cv_auc = cross_val_score(
        model, Xtr, ytr, cv=cv, scoring="roc_auc", n_jobs=-1
    ).mean()

    model.fit(Xtr, ytr)
    probabilities = model.predict_proba(Xte)[:, 1]

    results[name] = {
        "CV ROC-AUC": cv_auc,
        "Test ROC-AUC": roc_auc_score(yte, probabilities),
        "Test PR-AUC": average_precision_score(yte, probabilities)
    }

print(pd.DataFrame(results).T.round(3))

                    CV ROC-AUC  Test ROC-AUC  Test PR-AUC
LogisticRegression       0.984         0.986        0.981
RandomForest             0.986         0.989        0.983


In [51]:

p = model.predict_proba(Xte)[:, 1]


t = 0.50

y_pred = (p >= t).astype(int)

print(f"\nTest threshold = {t:.2f}")
print("Confusion matrix:")
print(confusion_matrix(yte, y_pred))

print("\nClassification report:")
print(classification_report(yte, y_pred, digits=3, zero_division=0))

print("Test ROC-AUC: %.3f" % roc_auc_score(yte, p))


tmp = pd.DataFrame({
    "y": np.asarray(yte),
    "p": p
})

tmp["decile"] = pd.qcut(
    tmp["p"].rank(method="first"),
    10,
    labels=False
) + 1

lift = (
    tmp.groupby("decile", observed=False)["y"]
       .agg(["count", "mean"])
       .rename(columns={"mean": "churn_rate"})
       .sort_index(ascending=False)
)

overall_churn_rate = tmp["y"].mean()
lift["lift"] = (
    lift["churn_rate"] / overall_churn_rate
    if overall_churn_rate > 0
    else np.nan
)

print("\nLift by decile (highest-risk group first):")
print(lift.round(3))

# Permutation importance on the test set is descriptive;
# use validation data for feature selection.
pi = permutation_importance(
    model,
    Xte,
    yte,
    scoring="roc_auc",
    n_repeats=10,
    random_state=SEED
)

imp = pd.Series(
    pi.importances_mean,
    index=Xte.columns
).sort_values(ascending=False)

print("\nTop drivers (permutation importance, AUC drop):")
print(imp.head(15).round(4))

imp.head(15)[::-1].plot.barh(
    figsize=(7, 5),
    title="Top churn drivers"
)
plt.tight_layout()
plt.savefig(f"{OUT}/feature_importance.png", dpi=130)
plt.close()



Test threshold = 0.50
Confusion matrix:
[[334   0]
 [  4 159]]

Classification report:
              precision    recall  f1-score   support

           0      0.988     1.000     0.994       334
           1      1.000     0.975     0.988       163

    accuracy                          0.992       497
   macro avg      0.994     0.988     0.991       497
weighted avg      0.992     0.992     0.992       497

Test ROC-AUC: 1.000

Lift by decile (highest-risk group first):
        count  churn_rate   lift
decile                          
10         50        1.00  3.049
9          50        1.00  3.049
8          49        1.00  3.049
7          50        0.28  0.854
6          49        0.00  0.000
5          50        0.00  0.000
4          50        0.00  0.000
3          49        0.00  0.000
2          50        0.00  0.000
1          50        0.00  0.000

Top drivers (permutation importance, AUC drop):
recency_m          0.0052
coupon_l6          0.0002
coupon_l3          0.000

In [55]:

# Train the final model on all labelled data
model.fit(X, y)

# Score customers
df["CHURN_PROB"] = model.predict_proba(X)[:, 1]

df["RISK_BAND"] = pd.cut(
    df["CHURN_PROB"],
    bins=[-0.01, 0.30, 0.60, 1.01],
    labels=["Low", "Medium", "High"]
)

df["VALUE_BAND"] = pd.qcut(
    df["spend_l12"].fillna(0).rank(method="first"),
    q=3,
    labels=["Low value", "Mid value", "High value"]
)

df["ACTION"] = np.select(
    [
        (df["RISK_BAND"] == "High") &
        (df["VALUE_BAND"] == "High value"),

        df["RISK_BAND"] == "High",

        (df["RISK_BAND"] == "Medium") &
        (df["complaints_l12"].fillna(0) > 0),

        df["RISK_BAND"] == "Medium"
    ],
    [
        "P1: personal outreach + win-back offer",
        "P2: automated win-back (coupon/email/app push)",
        "P3: service recovery call",
        "P4: nurture - engagement campaign"
    ],
    default="Monitor"
)

cols = [
    "CUSTOMER_ID", "CHURNED", "CHURN_PROB",
    "RISK_BAND", "VALUE_BAND", "ACTION",
    "MEMBERSHIP_TIER", "recency_m",
    "txn_l3", "spend_l12", "complaints_l12"
]

output_file = f"{OUT}/churn_scored_customers.xlsx"

# NOTE: These metrics are valid as independent test metrics only
# if model is still the model trained on the training split.
# Since it was just refitted on all X, do not report these as unbiased.
metrics_df = pd.DataFrame([{
    "Note": "Final model refitted on all labelled data; use previously saved held-out test metrics for evaluation."
}])

with pd.ExcelWriter(output_file) as xw:
    df[cols].sort_values(
        "CHURN_PROB", ascending=False
    ).to_excel(xw, sheet_name="scored", index=False)

    metrics_df.to_excel(
        xw, sheet_name="model_metrics", index=False
    )

    imp.rename("importance").to_frame().to_excel(
        xw, sheet_name="drivers"
    )

    lift.to_excel(xw, sheet_name="decile_lift")

    pd.crosstab(
        df["RISK_BAND"], df["VALUE_BAND"]
    ).to_excel(xw, sheet_name="risk_x_value")

print("Saved report:", output_file)
print("\nCustomer counts by risk and value:")
print(pd.crosstab(df["RISK_BAND"], df["VALUE_BAND"]))


Saved report: /kaggle/working/churn_output/churn_scored_customers.xlsx

Customer counts by risk and value:
VALUE_BAND  Low value  Mid value  High value
RISK_BAND                                   
Low               306        557         773
Medium             23         34          15
High              498        236          39
